### **Oxygenation Saturation Index (OSI)** $= \frac{Paw \space \times \space FiO_{2}}{SpO_{2}}$

In [1]:
# PARDS V3 12th-hr OSI Calculation
import os
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
input_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_12th_Raw/Study23_Tag285_EventList/RENAMED"
output_folder = os.path.join(input_folder, "OSI_V3_12th")
os.makedirs(output_folder, exist_ok=True)

metadata_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# -----------------------------
# Mapping from signal names to column names
# -----------------------------
signal_map = {
    "AVEA - Paw": "CAPSULE_AVEAA_VITAL_4729",
    "CDGR - Paw": "CAPSULE_DRAGERMEDIBUS_VITAL_1415",
    "SVU - Paw": "CAPSULE_MAQUETC_VITAL_1415",
    "AVEA - FiO₂": "CAPSULE_AVEAA_VITAL_2343",
    "CDGR - FiO₂": "CAPSULE_DRAGERMEDIBUS_VITAL_635",
    "SVU - FiO₂": "CAPSULE_MAQUETC_VITAL_635",
    "GE - SpO2 1": "PARM_SPO2_1"
}

# -----------------------------
# Load metadata Excel (Sheet3)
# -----------------------------
metadata_df = pd.read_excel(metadata_path, sheet_name="Sheet3")

# Ensure the key column exists
if "12th_rel_fname" not in metadata_df.columns:
    raise ValueError("Sheet3 is missing the required column: '12th_rel_fname'")

# Normalize key column (strip whitespace)
metadata_df["12th_rel_fname"] = metadata_df["12th_rel_fname"].astype(str).str.strip()

# Add columns for OSI stats if not already present
if "OSI_V3_12th_avg" not in metadata_df.columns:
    metadata_df["OSI_V3_12th_avg"] = pd.NA
if "OSI_V3_12th_std" not in metadata_df.columns:
    metadata_df["OSI_V3_12th_std"] = pd.NA

# Build a quick lookup: key -> row indices (handle duplicates safely)
# If duplicates exist, we'll update all matching rows.
key_to_indices = metadata_df.groupby("12th_rel_fname").indices

# -----------------------------
# Process each CSV file
# -----------------------------
spo2_col = signal_map["GE - SpO2 1"]
paw_cols = [signal_map["AVEA - Paw"], signal_map["CDGR - Paw"], signal_map["SVU - Paw"]]
fio2_cols = [signal_map["AVEA - FiO₂"], signal_map["CDGR - FiO₂"], signal_map["SVU - FiO₂"]]

for filename in os.listdir(input_folder):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(input_folder, filename)
    df = pd.read_csv(filepath)

    # Compute OSI using the first available (paw, fio2, spo2) set in the file
    osi_series = None

    # Coerce candidate columns to numeric when present
    if spo2_col in df.columns:
        df[spo2_col] = pd.to_numeric(df[spo2_col], errors="coerce")

    for paw in paw_cols:
        if paw not in df.columns:
            continue
        df[paw] = pd.to_numeric(df[paw], errors="coerce")

        for fio2 in fio2_cols:
            if fio2 not in df.columns or spo2_col not in df.columns:
                continue
            df[fio2] = pd.to_numeric(df[fio2], errors="coerce")

            denom = df[spo2_col].replace(0, pd.NA)
            temp_osi = (df[paw] * df[fio2]) / denom
            temp_osi = temp_osi.replace([float("inf"), -float("inf")], pd.NA).dropna()

            if len(temp_osi) > 0:
                osi_series = temp_osi
                # Save full OSI column aligned with original rows
                df["OSI"] = (df[paw] * df[fio2]) / denom
                break

        if osi_series is not None:
            break

    # Save updated CSV (always)
    df.to_csv(os.path.join(output_folder, filename), index=False)

    # -----------------------------
    # Update metadata: map filename -> 12th_rel_fname key
    # Example file: 1000780_20250420_18_12th_Raw.csv
    # base_name:     1000780_20250420_18_12th_Raw
    # key wanted:    1000780_20250420_18_12th
    # -----------------------------
    base_name = filename[:-4]               # remove ".csv"
    key = base_name.replace("_Raw", "")     # remove "_Raw" -> matches 12th_rel_fname

    if osi_series is not None and key in key_to_indices:
        idx_list = key_to_indices[key]
        metadata_df.loc[idx_list, "OSI_V3_12th_avg"] = float(osi_series.mean())
        metadata_df.loc[idx_list, "OSI_V3_12th_std"] = float(osi_series.std())

# -----------------------------
# Save updated Excel to Sheet4 (preserve other sheets)
# -----------------------------
with pd.ExcelWriter(metadata_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    metadata_df.to_excel(writer, sheet_name="Sheet4", index=False)

print(f"Updated CSVs saved to: {output_folder}")
print(f"Updated Excel written to Sheet4 in: {metadata_path}")


Updated CSVs saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_12th_Raw/Study23_Tag285_EventList/RENAMED/OSI_V3_12th
Updated Excel written to Sheet4 in: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx
